In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
API_KEY = os.getenv("EIA_API_KEY")
assert API_KEY, "No key found"

In [2]:
URL = "https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/"

In [3]:
params = {
    "api_key": API_KEY,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": "CISO",
    "start": "2025-10-01T00",
    "end": "2025-10-02T00",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000,
}

r = requests.get(URL, params=params, timeout=30)
r.raise_for_status()
payload = r.json()

In [4]:
print(payload.keys())
print(payload["response"].keys())
print(payload["response"]["data"][1])

dict_keys(['response', 'request', 'apiVersion', 'ExcelAddInVersion'])
dict_keys(['total', 'dateFormat', 'frequency', 'data', 'description'])
{'period': '2025-10-01T00', 'respondent': 'CISO', 'respondent-name': 'California Independent System Operator', 'fueltype': 'NG', 'type-name': 'Natural Gas', 'value': '9431', 'value-units': 'megawatthours'}


In [5]:
rows = payload["response"]["data"]
df = pd.DataFrame(rows)

df["value"] = pd.to_numeric(df["value"], errors = "coerce")
df["period"] = pd.to_datetime(df["period"], format="%Y-%m-%dT%H", utc=True)

print(df.shape)
print(df.dtypes)

df.head(10)

(200, 7)
period             datetime64[ns, UTC]
respondent                      object
respondent-name                 object
fueltype                        object
type-name                       object
value                            int64
value-units                     object
dtype: object


,period,respondent,respondent-name,fueltype,type-name,value,value-units
0,2025-10-01 00:00:00+00:00,CISO,California Independent System Operator,COL,Coal,0,megawatthours
1,2025-10-01 00:00:00+00:00,CISO,California Independent System Operator,NG,Natural Gas,9431,megawatthours
2,2025-10-01 00:00:00+00:00,CISO,California Independent System Operator,NUC,Nuclear,2240,megawatthours
3,2025-10-01 00:00:00+00:00,CISO,California Independent System Operator,OIL,Petroleum,46,megawatthours
4,2025-10-01 00:00:00+00:00,CISO,California Independent System Operator,OTH,Other,-2471,megawatthours
5,2025-10-01 00:00:00+00:00,CISO,California Independent System Operator,SUN,Solar,15288,megawatthours
6,2025-10-01 00:00:00+00:00,CISO,California Independent System Operator,WAT,Hydro,729,megawatthours
7,2025-10-01 00:00:00+00:00,CISO,California Independent System Operator,WND,Wind,3711,megawatthours
8,2025-10-01 01:00:00+00:00,CISO,California Independent System Operator,COL,Coal,0,megawatthours
9,2025-10-01 01:00:00+00:00,CISO,California Independent System Operator,NG,Natural Gas,8710,megawatthours
